# 06 — Response Streaming

Замість очікування 10-30с на повну відповідь — показуємо текст
шматками, по мірі генерації.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

import sys
sys.path.append("..")
from helpers.chat_utils import client, model, add_user_message

print("Готово. client і model імпортовані напряму — тут не використовуємо chat(),\n"
      "бо стрімінг має свою окрему структуру виклику (client.messages.stream)")

## Базовий варіант — сирі events

Побач усі 6 типів подій наживо: MessageStart → ContentBlockStart →
ContentBlockDelta (багато разів) → ContentBlockStop → MessageDelta → MessageStop.

In [ ]:
messages = []
add_user_message(messages, "Write a 1 sentence description of a fake database")

stream = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    stream=True,
)

for event in stream:
    print(type(event).__name__)  # назва класу події — звір із таблицею в конспекті

## Спрощений варіант — тільки текст

SDK сам фільтрує все, крім ContentBlockDelta-тексту — саме те, що
зазвичай треба показати юзеру.

In [ ]:
messages2 = []
add_user_message(messages2, "Write a 1 sentence description of a fake database")

with client.messages.stream(
    model=model,
    max_tokens=1000,
    messages=messages2,
) as stream:
    for text in stream.text_stream:
        print(text, end="")  # end="" — щоб текст "друкувався" без переносів рядка

## Повне повідомлення після стрімінгу

Для збереження в БД / подальшої обробки — after the fact, коли
стрімінг уже закінчився.

In [ ]:
messages3 = []
add_user_message(messages3, "Write a 1 sentence description of a fake database")

with client.messages.stream(
    model=model,
    max_tokens=1000,
    messages=messages3,
) as stream:
    for text in stream.text_stream:
        print(text, end="")

    final_message = stream.get_final_message()

print("\n\n--- Повний message-об'єкт ---")
print(final_message)

## 🧪 Своя перевірка

Постав своє питання (бажано таке, де відповідь довша — краще видно ефект стрімінгу).

In [ ]:
my_messages = []
add_user_message(my_messages, "Твоє питання тут (краще щось на кілька речень)")

with client.messages.stream(
    model=model,
    max_tokens=1000,
    messages=my_messages,
) as stream:
    for text in stream.text_stream:
        print(text, end="")